In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
orig_df = pd.read_csv("bank_edited.csv")
orig_df.head(10)

In [ ]:
# Plot heatmap to visualize locations of missing values
sns.heatmap(orig_df.isnull(), yticklabels=False, cbar=False, cmap='viridis')

In [ ]:
orig_df.shape

In [ ]:
orig_df.columns

In [ ]:
orig_df['y'] = orig_df['y'].map({'yes': 1, 'no': 0})
orig_df['y'].value_counts()

In [ ]:
# Bar graph example
selected_var = 'y'
y = orig_df[selected_var].value_counts()
x_arr = y.index.tolist()

cmap = plt.cm.Set1
colors = cmap(np.linspace(0, 1, len(x_arr)))

fig, ax = plt.subplots()
ax.bar(x_arr, y, color=colors)

ax.set_xlabel(selected_var)
ax.set_ylabel('Frequency')
ax.set_title('Frequency of ' + selected_var)

#plt.tight_layout()
#plt.savefig('season_freq_bar.jpg')

plt.show()
# See more about bar graph at https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html

In [ ]:
orig_y = orig_df['y']
orig_y

In [ ]:
df = orig_df.copy()

In [ ]:
orig_X = df.drop(['y'], axis=1)
orig_X.columns

In [ ]:
# Finding categorical variables
cat_features = orig_X.select_dtypes(include=["object", "category"]).columns
cat_features

In [ ]:
# Finding numerical variables
num_features = orig_X.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_features

In [ ]:
num_orig_X = df[num_features]
num_orig_X.columns

### Scaling

In [ ]:
# Scale by standardized normal distribution, (x-mean)/sd
def scale_standard(X_train, X_test):
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by min-max, (x-min)/(max-min)
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by robust, (x-median)/iqr
def scale_robust(X_train, X_test):
    from sklearn.preprocessing import RobustScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

### Encoding Categorical Data

In [ ]:
# One-Hot Encoder version 1: convert all categorical variables
def onehot_encoder_all(df):
    from sklearn.preprocessing import OneHotEncoder
    onehot_encoder = OneHotEncoder(sparse_output=False,        # use sparse=True for large data
                                    handle_unknown='ignore')
    oh_encoded = onehot_encoder.fit_transform(df)
    ohe_df = pd.DataFrame(oh_encoded,
                         columns=onehot_encoder.get_feature_names_out(df.columns))
    return ohe_df

In [ ]:
# One-Hot Encoder version 2: Convert selected variables
def onehot_encoder_selected(df, encoded_vars):
    df = pd.get_dummies(df, columns=encoded_vars, drop_first=False)
    return df

In [ ]:
# Label Encoder
def label_encoder(df, encoded_var):
    from sklearn.preprocessing import LabelEncoder
    label_encoder = LabelEncoder()
    df[encoded_var] = label_encoder.fit_transform(df[encoded_var])
    return df

In [ ]:
# Ordinal Encoder
def ordinal_encoder(df, encoded_var, encode_order):
    from sklearn.preprocessing import OrdinalEncoder
    ordinal_encoder = OrdinalEncoder(categories=[encode_order])
    df[encoded_var] = ordinal_encoder.fit_transform(df[[encoded_var]])
    return df

In [ ]:
# Frequenct Encoder
def frequent_encoder(df, encoded_var):
    freq_encoding = df[encoded_var].value_counts() 
    df[encoded_var] = df[encoded_var].map(freq_encoding)
    return df

In [ ]:
# Target Encoder
def target_encoder(df, encoded_var, encoded_num):
    import category_encoders as ce
    target_encoder = ce.TargetEncoder(cols=[encoded_var])
    df[encoded_var] = target_encoder.fit_transform(df[encoded_var], df[encoded_num])
    return df

In [ ]:
# Binary Encoder
def binary_encoder(df, encoded_var):
    import category_encoders as ce
    binary_encoder = ce.BinaryEncoder(cols=[encoded_var])
    df_binary = binary_encoder.fit_transform(df[encoded_var])
    df_binary.columns = [f"{encoded_var}_bin_{i}" for i in range(df_binary.shape[1])]
    df = pd.concat([df, df_binary], axis=1)
    df = df.drop([encoded_var], axis=1)
    return df 

In [ ]:
### Hash Encoder
def hash_encoder(df, encoded_var, hash_len):
    import category_encoders as ce
    hash_encoder = ce.HashingEncoder(cols=[encoded_var], n_components=hash_len)  # n_components = length of hash, default = 8
    df_hash = hash_encoder.fit_transform(df[encoded_var])
    df_hash.columns = [f"{encoded_var}_hash_{i}" for i in range(df_hash.shape[1])]
    df = pd.concat([df, df_hash], axis=1)
    df = df.drop([encoded_var], axis=1)
    return df 

### Handling Imbalance

In [ ]:
# Undersampling
def undersampling(X,y):
    from imblearn.under_sampling import RandomUnderSampler
    X_res, y_res = RandomUnderSampler().fit_resample(X, y)
    return X_res, y_res

In [ ]:
# Oversampling
def oversampling(X,y):
    from imblearn.over_sampling import RandomOverSampler
    X_res, y_res = RandomOverSampler().fit_resample(X, y)
    return X_res, y_res

In [ ]:
# SMOTE
def smote(X,y):
    from imblearn.over_sampling import SMOTE
    X_res, y_res = SMOTE().fit_resample(X, y)
    return X_res, y_res

In [ ]:
### See class weight example in Experiment problems

### Performance Evaluation

In [ ]:
# Function to compute performance classification metrics
def evaluate_performance(y_true, y_pred, average_val='macro'):

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average=average_val)
    recall = recall_score(y_true, y_pred, average=average_val)
    f1 = f1_score(y_true, y_pred, average=average_val)

    from sklearn.metrics import confusion_matrix
    confusion = confusion_matrix(y_true, y_pred)

    return accuracy, precision, recall, f1, confusion

In [ ]:
def plot_roc_auc(y_true, y_pred):
    from sklearn.metrics import roc_curve, roc_auc_score
    import matplotlib.pyplot as plt

    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred)

    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (AUC = {auc:.3f})")
    plt.show()

    return auc

### Experiments

#### Baseline model

In [ ]:
X = num_orig_X.copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Scaling
X_train, X_test = scale_standard(X_train, X_test)

# Construct a model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

### Encoding Categorical Data 

In [ ]:
### >>> Let's use all features (both numerical and categoical features) to build the model

In [ ]:
X = orig_X.copy()  
y = orig_y.copy()

X = onehot_encoder_all(X)
X.head()

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
X_train, X_test = scale_standard(X_train, X_test)

# Construct a model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
### Question: How many columns of features before and after one-hot encoding?
### Answer:

In [ ]:
### >>> Instruction: Let's add only 'marital', 'education' to help with model learning

In [ ]:
num_features

In [ ]:
encoded_vars = ['marital', 'education']
X = orig_X[num_features+encoded_vars].copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = onehot_encoder_selected(X_train, encoded_vars)
X_test = onehot_encoder_selected(X_test, encoded_vars)
X_train.head()

In [ ]:
# Scaling
X_train, X_test = scale_standard(X_train, X_test)

# Construct a model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
### Question: How many columns of features before and after one-hot encoding only 'marital', 'education'?
### Answer:

In [ ]:
encoded_vars = ['education']
X = orig_X[num_features+encoded_vars].copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

############## Categorical Data Encoding ##############
## Label Encoder
encoded_var = 'education'
X_train = label_encoder(X_train, encoded_var)
X_test = label_encoder(X_test, encoded_var)
X_train.head()
#######################################################

# Scaling
X_train, X_test = scale_standard(X_train, X_test)

# Construct a model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
### Ordinal Encoder Example
encoded_var = 'education'
encode_order = ['unknown', 'primary', 'secondary', 'tertiary']  # Item1 = 0, Item2= 1, ... 
X_train = ordinal_encoder(X_train, encoded_var, encode_order)
X_test = ordinal_encoder(X_test, encoded_var, encode_order)
X_train.head()

In [ ]:
### Frequenct Encoder Example
encoded_var = 'education'
X_train = frequent_encoder(X_train, encoded_var)
X_test = frequent_encoder(X_test, encoded_var)
X_train.head()

In [ ]:
### Target Encoder Example
encoded_var = 'education'
encoded_num = 'balance'
X_train = target_encoder(X_train, encoded_var, encoded_num)
X_test = target_encoder(X_test, encoded_var, encoded_num)
X_train.head()

In [ ]:
### Binary Encoder Example
encoded_var = 'education'
X_train = binary_encoder(X_train, encoded_var)
X_test = binary_encoder(X_test, encoded_var)
X_train.head()

In [ ]:
### Hash Encoding Example
encoded_var = 'education'
hash_len = 4  # #bits to represent size of education
X_train = hash_encoder(X_train, encoded_var, hash_len)
X_test = hash_encoder(X_test, encoded_var, hash_len)
X_train.head()

In [ ]:
encoded_vars = ['education']
X = orig_X[num_features+encoded_vars].copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

############## Replace Categorical Data Encoding here ##############
encoded_var = 'education'
X_train = label_encoder(X_train, encoded_var)
X_test = label_encoder(X_test, encoded_var)
X_train.head()
####################################################################

# Scaling
X_train, X_test = scale_standard(X_train, X_test)

# Construct a model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
### Question: Not including one-hot encoding, which categorical data encoding gives the best performance?
### Instruction: Show your work and give the answer below
### Annswer: 

In [ ]:
### Question: Try (1) another categorical data and 
###               (2) some encoding methods 
###           that results to the better performance than numerical features + education feature.
### Instruction: Show your work and give the answer below
### Annswer: 

### Imbalance Handling

In [ ]:
# Undersampling
X = num_orig_X.copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

############## Replace Imbalance Handling here ##############
X_train, y_train = undersampling(X_train,y_train)
print(X_train.shape, y_train.shape)
y_train.value_counts()
#############################################################

# Scaling
X_train, X_test = scale_standard(X_train, X_test)

# Construct a model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
# Oversampling Example
X_train, y_train = oversampling(X_train,y_train)
print(X_train.shape, y_train.shape)
y_train.value_counts()

# Smote Example
X_train, y_train = smote(X_train,y_train)
print(X_train.shape, y_train.shape)
y_train.value_counts()

In [ ]:
### Class Weight Example
X = num_orig_X.copy()
y = orig_y.copy()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class_weight_dict = {0: 1.0,   # majority
                     1: 20.0   # minority
                   }
print(class_weight_dict)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight=class_weight_dict, max_iter=1000)
model.fit(X_train, y_train)

# Use model for predicted responses
y_pred = model.predict(X_test)

# Measure performances
auc = plot_roc_auc(y_test, y_pred)
print(auc)

acc, precision, recall, f1, confusion = evaluate_performance(y_test, y_pred)
print(confusion)
print(acc, precision, recall, f1)

In [ ]:
### Question: From the previous model (the one that gives better performance than numerical features + education feature)
###           Apply the imbalance handling and find out which imbalance handling methods gives the best performance
### Instruction: Show your work and give the answer below
### Annswer: 